<a href="https://colab.research.google.com/github/Light466/My_Programs/blob/main/project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# DYNAMIC BLOCKCHAIN-BASED SECURE HOSPITAL EHR MANAGEMENT SYSTEM
# =====================================================================
#
# GOOGLE COLAB VERSION
#
# No Tkinter
# No desktop GUI
#
# Uses:
#   IPyWidgets browser interface
#   SHA-256 authentication
#   Dynamic 256-bit session key
#   Dynamic encryption key
#   Dynamic authentication token
#   Dynamic blockchain
#   Existing blockchain_db.json
#   Existing audit_log.json
#   Medicine JSON repository
#   Symptoms JSON repository
#   Medicine sentiment CSV
#   Patient EHR management
#   CSV export
#   Blockchain verification
#
# =====================================================================


# =====================================================================
# 1. IMPORT LIBRARIES
# =====================================================================

import os
import json
import hashlib
import secrets
import base64
import datetime
import time
import io
import re

import pandas as pd
import ipywidgets as widgets

from IPython.display import display, clear_output, HTML

from google.colab import files


# =====================================================================
# 2. GOOGLE COLAB WORKING DIRECTORY
# =====================================================================

BASE_DIR = "/content"

BLOCKCHAIN_FILE = os.path.join(
    BASE_DIR,
    "blockchain_db.json"
)

AUDIT_FILE = os.path.join(
    BASE_DIR,
    "audit_log.json"
)

PATIENT_FILE = os.path.join(
    BASE_DIR,
    "patient_records.csv"
)


# =====================================================================
# 3. USER DATABASE
# =====================================================================

# Passwords are NEVER stored as plain text.
#
# doctor1  -> Doctor@123
# admin    -> Admin@123
#
# SHA-256 hashes are generated here.

USERS = {

    "doctor1":
        hashlib.sha256(
            "Doctor@123".encode("utf-8")
        ).hexdigest(),

    "admin":
        hashlib.sha256(
            "Admin@123".encode("utf-8")
        ).hexdigest()
}


# =====================================================================
# 4. HELPER FUNCTIONS
# =====================================================================

def sha256_text(text):

    return hashlib.sha256(
        str(text).encode("utf-8")
    ).hexdigest()


def hash_password(password):

    return sha256_text(password)


def now_string():

    return datetime.datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )


# =====================================================================
# 5. UPLOAD THE USER'S ACTUAL FILES
# =====================================================================
#
# In Google Colab, the uploaded files are copied into /content.
#
# The program accepts:
#
#   blockchain_db(9).json
#   audit_log(9).json
#   mydb.Medicines*.json
#   mydb.Symptoms*.json
#   medicine_sentiment_results_with_actual.csv
#
# =====================================================================

print("=" * 80)
print("DYNAMIC SECURE HOSPITAL EHR")
print("=" * 80)

print()
print("Upload your existing hospital files.")
print()
print("Required:")
print("1. blockchain_db JSON")
print("2. audit_log JSON")
print("3. Medicine JSON")
print("4. Symptoms JSON")
print("5. Medicine sentiment CSV")
print()

uploaded_files = files.upload()


# =====================================================================
# 6. COPY UPLOADED FILES TO STANDARD COLAB NAMES
# =====================================================================

def find_uploaded_file(
        keywords,
        extension=None
):

    for filename in uploaded_files.keys():

        lower_name = filename.lower()

        matched = True

        for keyword in keywords:

            if keyword.lower() not in lower_name:

                matched = False
                break

        if extension:

            if not lower_name.endswith(
                extension.lower()
            ):

                matched = False

        if matched:

            return filename

    return None


# ---------------------------------------------------------------------
# Blockchain
# ---------------------------------------------------------------------

blockchain_uploaded = find_uploaded_file(
    ["blockchain"],
    ".json"
)

if blockchain_uploaded:

    with open(
        BLOCKCHAIN_FILE,
        "wb"
    ) as f:

        f.write(
            uploaded_files[
                blockchain_uploaded
            ]
        )


# ---------------------------------------------------------------------
# Audit log
# ---------------------------------------------------------------------

audit_uploaded = find_uploaded_file(
    ["audit"],
    ".json"
)

if audit_uploaded:

    with open(
        AUDIT_FILE,
        "wb"
    ) as f:

        f.write(
            uploaded_files[
                audit_uploaded
            ]
        )


# ---------------------------------------------------------------------
# Medicine JSON
# ---------------------------------------------------------------------

medicine_uploaded = find_uploaded_file(
    ["medicine"],
    ".json"
)

if medicine_uploaded:

    medicine_file = os.path.join(
        BASE_DIR,
        "mydb.Medicines.json"
    )

    with open(
        medicine_file,
        "wb"
    ) as f:

        f.write(
            uploaded_files[
                medicine_uploaded
            ]
        )

else:

    medicine_file = None


# ---------------------------------------------------------------------
# Symptoms JSON
# ---------------------------------------------------------------------

symptoms_uploaded = find_uploaded_file(
    ["symptom"],
    ".json"
)

if symptoms_uploaded:

    symptoms_file = os.path.join(
        BASE_DIR,
        "mydb.Symptoms.json"
    )

    with open(
        symptoms_file,
        "wb"
    ) as f:

        f.write(
            uploaded_files[
                symptoms_uploaded
            ]
        )

else:

    symptoms_file = None


# ---------------------------------------------------------------------
# Sentiment CSV
# ---------------------------------------------------------------------

sentiment_uploaded = find_uploaded_file(
    ["sentiment"],
    ".csv"
)

if sentiment_uploaded:

    sentiment_file = os.path.join(
        BASE_DIR,
        "medicine_sentiment_results_with_actual.csv"
    )

    with open(
        sentiment_file,
        "wb"
    ) as f:

        f.write(
            uploaded_files[
                sentiment_uploaded
            ]
        )

else:

    sentiment_file = None


# =====================================================================
# 7. INITIALIZE BLOCKCHAIN IF NECESSARY
# =====================================================================

def initialize_blockchain():

    if not os.path.exists(
        BLOCKCHAIN_FILE
    ):

        timestamp = time.time()

        genesis_data = {
            "msg": "Genesis"
        }

        previous_hash = "0"

        hash_input = (
            str(0)
            +
            json.dumps(
                genesis_data,
                sort_keys=True
            )
            +
            str(timestamp)
            +
            previous_hash
        )

        block_hash = sha256_text(
            hash_input
        )

        genesis_block = {

            "index": 0,

            "data":
                genesis_data,

            "timestamp":
                timestamp,

            "previous_hash":
                previous_hash,

            "hash":
                block_hash
        }

        with open(
            BLOCKCHAIN_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                [genesis_block],
                f,
                indent=4
            )


# =====================================================================
# 8. LOAD BLOCKCHAIN
# =====================================================================

def load_blockchain():

    initialize_blockchain()

    try:

        with open(
            BLOCKCHAIN_FILE,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        if isinstance(data, list):

            return data

        return []

    except Exception:

        return []


# =====================================================================
# 9. SAVE BLOCKCHAIN
# =====================================================================

def save_blockchain(
        blockchain
):

    with open(
        BLOCKCHAIN_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            blockchain,
            f,
            indent=4,
            ensure_ascii=False
        )


# =====================================================================
# 10. CREATE DYNAMIC BLOCKCHAIN BLOCK
# =====================================================================

def create_block(
        block_data
):

    blockchain = load_blockchain()

    if len(blockchain) == 0:

        index = 0

        previous_hash = "0"

    else:

        index = (
            blockchain[-1].get(
                "index",
                len(blockchain) - 1
            )
            + 1
        )

        previous_hash = (
            blockchain[-1].get(
                "hash",
                "0"
            )
        )


    timestamp = time.time()


    data_string = json.dumps(

        block_data,

        sort_keys=True,

        ensure_ascii=False
    )


    hash_input = (

        str(index)

        +

        data_string

        +

        str(timestamp)

        +

        previous_hash
    )


    block_hash = sha256_text(
        hash_input
    )


    block = {

        "index":
            index,

        "data":
            block_data,

        "timestamp":
            timestamp,

        "previous_hash":
            previous_hash,

        "hash":
            block_hash
    }


    blockchain.append(
        block
    )


    save_blockchain(
        blockchain
    )


    return block


# =====================================================================
# 11. LOAD AUDIT LOG
# =====================================================================

def load_audit_log():

    if not os.path.exists(
        AUDIT_FILE
    ):

        return []


    try:

        with open(
            AUDIT_FILE,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)

        if isinstance(data, list):

            return data

        return []

    except Exception:

        return []


# =====================================================================
# 12. SAVE AUDIT LOG
# =====================================================================

def save_audit_log(
        logs
):

    with open(
        AUDIT_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            logs,
            f,
            indent=4,
            ensure_ascii=False
        )


# =====================================================================
# 13. ADD AUDIT LOG
# =====================================================================

def add_audit_log(
        username,
        action
):

    logs = load_audit_log()


    logs.append({

        "user":
            username,

        "action":
            action,

        "time":
            time.time(),

        "datetime":
            now_string()
    })


    save_audit_log(
        logs
    )


# =====================================================================
# 14. LOAD MEDICINE JSON
# =====================================================================

def load_medicines():

    if not medicine_file:

        return []


    try:

        with open(
            medicine_file,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)


        if isinstance(data, list):

            return data

        return []

    except Exception as e:

        print(
            "Medicine JSON error:",
            e
        )

        return []


medicines = load_medicines()


# =====================================================================
# 15. LOAD SYMPTOMS JSON
# =====================================================================

def load_symptoms():

    if not symptoms_file:

        return []


    try:

        with open(
            symptoms_file,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)


        if isinstance(data, list):

            return data

        return []

    except Exception as e:

        print(
            "Symptoms JSON error:",
            e
        )

        return []


symptoms = load_symptoms()


# =====================================================================
# 16. LOAD SENTIMENT CSV
# =====================================================================

def load_sentiment():

    if not sentiment_file:

        return pd.DataFrame()


    try:

        df = pd.read_csv(
            sentiment_file
        )

        df.columns = (
            df.columns
            .astype(str)
            .str.strip()
        )

        return df

    except Exception as e:

        print(
            "Sentiment CSV error:",
            e
        )

        return pd.DataFrame()


sentiment_df = load_sentiment()


# =====================================================================
# 17. PREPARE MEDICINE DICTIONARY
# =====================================================================

medicine_dict = {}


for record in medicines:

    name = record.get(
        "Name"
    )


    if name is None:

        continue


    name = str(
        name
    ).strip()


    if not name:

        continue


    medicine_dict[
        name
    ] = record


medicine_names = sorted(
    medicine_dict.keys(),
    key=lambda x: x.lower()
)


# =====================================================================
# 18. PREPARE SYMPTOM / DISEASE DICTIONARY
# =====================================================================

disease_dict = {}


for record in symptoms:

    disease_name = (

        record.get(
            "D_Name"
        )

        or

        record.get(
            "Name"
        )
    )


    if disease_name:

        disease_name = str(
            disease_name
        ).strip()


        disease_dict[
            disease_name
        ] = record


disease_names = sorted(
    disease_dict.keys(),
    key=lambda x: x.lower()
)


# =====================================================================
# 19. PREPARE SENTIMENT DICTIONARY
# =====================================================================

sentiment_dict = {}


if not sentiment_df.empty:

    if "Name" in sentiment_df.columns:

        for _, row in sentiment_df.iterrows():

            name = row.get(
                "Name"
            )


            if pd.isna(name):

                continue


            name = str(
                name
            ).strip()


            if not name:

                continue


            sentiment_dict[
                name.lower()
            ] = {

                "score":
                    row.get(
                        "SentimentScore",
                        ""
                    ),

                "label":
                    row.get(
                        "SentimentLabel",
                        ""
                    ),

                "actual":
                    row.get(
                        "Actual_Sentiment",
                        ""
                    ),

                "matched_disease":
                    row.get(
                        "MatchedDisease",
                        ""
                    )
            }


# =====================================================================
# 20. CURRENT SECURITY SESSION
# =====================================================================

CURRENT_USER = None

CURRENT_SESSION_KEY = None

CURRENT_ENCRYPTION_KEY = None

CURRENT_TOKEN = None


# =====================================================================
# 21. DYNAMIC 256-BIT SESSION KEY
# =====================================================================

def generate_session_key():

    # 32 bytes = 256 bits

    return secrets.token_bytes(
        32
    )


# =====================================================================
# 22. DYNAMIC ENCRYPTION KEY
# =====================================================================

def generate_encryption_key(
        session_key
):

    return base64.urlsafe_b64encode(
        session_key
    ).decode(
        "utf-8"
    )


# =====================================================================
# 23. DYNAMIC AUTHENTICATION TOKEN
# =====================================================================

def generate_token(
        username,
        session_key
):

    nonce = secrets.token_bytes(
        32
    )


    timestamp = (
        str(
            time.time()
        )
    )


    token_input = (

        username

        +

        timestamp

        +

        nonce.hex()
    )


    token = sha256_text(

        session_key.hex()
        +

        token_input
    )


    return token


# =====================================================================
# 24. EHR ENCRYPTION DEMONSTRATION
# =====================================================================

def encrypt_ehr(
        ehr_text,
        session_key
):

    ehr_bytes = (
        ehr_text
        .encode(
            "utf-8"
        )
    )


    encrypted = bytes(

        [

            ehr_bytes[i]
            ^
            session_key[
                i % len(
                    session_key
                )
            ]

            for i in range(
                len(ehr_bytes)
            )
        ]
    )


    return base64.b64encode(
        encrypted
    ).decode(
        "utf-8"
    )


# =====================================================================
# 25. VERIFY BLOCKCHAIN
# =====================================================================

def verify_blockchain():

    blockchain = load_blockchain()

    results = []

    valid = True


    for i, block in enumerate(
        blockchain
    ):

        index = block.get(
            "index"
        )

        data = block.get(
            "data",
            {}
        )

        timestamp = block.get(
            "timestamp"
        )

        previous_hash = block.get(
            "previous_hash"
        )

        stored_hash = block.get(
            "hash"
        )


        hash_input = (

            str(index)

            +

            json.dumps(
                data,
                sort_keys=True,
                ensure_ascii=False
            )

            +

            str(timestamp)

            +

            str(previous_hash)
        )


        calculated_hash = sha256_text(
            hash_input
        )


        hash_ok = (
            calculated_hash
            ==
            stored_hash
        )


        previous_ok = True


        if i > 0:

            previous_ok = (

                previous_hash

                ==

                blockchain[
                    i - 1
                ].get(
                    "hash"
                )
            )


        block_ok = (
            hash_ok
            and
            previous_ok
        )


        if not block_ok:

            valid = False


        results.append({

            "Block":
                index,

            "Hash Valid":
                hash_ok,

            "Previous Hash Valid":
                previous_ok,

            "Status":
                "VALID"
                if block_ok
                else "TAMPERED"
        })


    return valid, pd.DataFrame(
        results
    )


# =====================================================================
# 26. LOGIN INTERFACE
# =====================================================================

login_header = HTML(

    """
    <div style="
        background:linear-gradient(
            90deg,
            #17365d,
            #1f4e78
        );
        color:white;
        padding:25px;
        border-radius:12px;
        text-align:center;
        margin-bottom:20px;
    ">

        <h1>
            🔐 SECURE HOSPITAL SYSTEM
        </h1>

        <h3>
            Dynamic Blockchain-Based Secure EHR
        </h3>

        <p>
            SHA-256 Authentication |
            Dynamic 256-bit Session |
            Blockchain Security
        </p>

    </div>
    """
)


login_user = widgets.Text(

    description="Username:",

    placeholder="doctor1",

    layout=widgets.Layout(
        width="450px"
    )
)


login_password = widgets.Password(

    description="Password:",

    placeholder="Password",

    layout=widgets.Layout(
        width="450px"
    )
)


login_button = widgets.Button(

    description="LOGIN",

    button_style="success",

    icon="sign-in",

    layout=widgets.Layout(
        width="180px"
    )
)


login_output = widgets.Output()


# =====================================================================
# 27. LOGIN FUNCTION
# =====================================================================

def perform_login(
        button
):

    global CURRENT_USER

    global CURRENT_SESSION_KEY

    global CURRENT_ENCRYPTION_KEY

    global CURRENT_TOKEN


    with login_output:

        clear_output()


        username = (
            login_user.value
            .strip()
        )


        password = (
            login_password.value
        )


        # -------------------------------------------------------------
        # CHECK USERNAME
        # -------------------------------------------------------------

        if username not in USERS:

            add_audit_log(
                username
                if username
                else "Unknown",
                "Failed Login"
            )


            print()
            print(
                "❌ INVALID USERNAME OR PASSWORD"
            )

            return


        # -------------------------------------------------------------
        # CHECK PASSWORD
        # -------------------------------------------------------------

        entered_hash = hash_password(
            password
        )


        if entered_hash != USERS[
            username
        ]:

            add_audit_log(
                username,
                "Failed Login"
            )


            print()
            print(
                "❌ INVALID USERNAME OR PASSWORD"
            )

            return


        # -------------------------------------------------------------
        # GENERATE DYNAMIC SESSION KEY
        # -------------------------------------------------------------

        CURRENT_USER = username


        CURRENT_SESSION_KEY = (
            generate_session_key()
        )


        # -------------------------------------------------------------
        # GENERATE DYNAMIC ENCRYPTION KEY
        # -------------------------------------------------------------

        CURRENT_ENCRYPTION_KEY = (

            generate_encryption_key(

                CURRENT_SESSION_KEY
            )
        )


        # -------------------------------------------------------------
        # GENERATE DYNAMIC TOKEN
        # -------------------------------------------------------------

        CURRENT_TOKEN = generate_token(

            username,

            CURRENT_SESSION_KEY
        )


        # -------------------------------------------------------------
        # TOKEN FINGERPRINT
        # -------------------------------------------------------------

        token_fingerprint = sha256_text(
            CURRENT_TOKEN
        )


        # -------------------------------------------------------------
        # CREATE BLOCKCHAIN LOGIN BLOCK
        # -------------------------------------------------------------

        login_data = {

            "type":
                "USER_LOGIN",

            "username":
                username,

            "authentication":
                "SHA-256",

            "session_key_bits":
                256,

            "token_fingerprint":
                token_fingerprint,

            "login_time":
                now_string()
        }


        login_block = create_block(
            login_data
        )


        # -------------------------------------------------------------
        # AUDIT LOG
        # -------------------------------------------------------------

        add_audit_log(

            username,

            "login"
        )


        # -------------------------------------------------------------
        # LOGIN SUCCESS MESSAGE
        # -------------------------------------------------------------

        print(
            "=" * 80
        )

        print(
            "✅ LOGIN SUCCESSFUL"
        )

        print(
            "=" * 80
        )

        print()

        print(
            "User:"
        )

        print(
            username
        )

        print()

        print(
            "SHA-256 Password Hash:"
        )

        print(
            USERS[
                username
            ]
        )

        print()

        print(
            "Dynamic 256-bit Session Key:"
        )

        print(
            CURRENT_SESSION_KEY.hex()
        )

        print()

        print(
            "Dynamic Encryption Key:"
        )

        print(
            CURRENT_ENCRYPTION_KEY
        )

        print()

        print(
            "Dynamic Authentication Token:"
        )

        print(
            CURRENT_TOKEN
        )

        print()

        print(
            "Token Fingerprint:"
        )

        print(
            token_fingerprint
        )

        print()

        print(
            "Blockchain Block Index:"
        )

        print(
            login_block[
                "index"
            ]
        )

        print()

        print(
            "Blockchain Hash:"
        )

        print(
            login_block[
                "hash"
            ]
        )

        print(
            "=" * 80
        )


        # -------------------------------------------------------------
        # OPEN HOSPITAL APPLICATION
        # -------------------------------------------------------------

        display_hospital_application()


login_button.on_click(
    perform_login
)


# =====================================================================
# 28. DISPLAY LOGIN WINDOW
# =====================================================================

display(
    login_header,

    widgets.VBox([

        login_user,

        login_password,

        login_button

    ]),

    login_output
)


# =====================================================================
# 29. HOSPITAL APPLICATION
# =====================================================================

def display_hospital_application():

    clear_output(
        wait=True
    )


    # =================================================================
    # SESSION DATA
    # =================================================================

    patient_records = []


    # =================================================================
    # HEADER
    # =================================================================

    display(

        HTML(

            f"""
            <div style="
                background:linear-gradient(
                    90deg,
                    #17365d,
                    #1f4e78
                );
                color:white;
                padding:25px;
                border-radius:12px;
                text-align:center;
            ">

                <h1>
                    🏥 SECURE HOSPITAL MANAGEMENT SYSTEM
                </h1>

                <p>
                    Logged in user:
                    <b>{CURRENT_USER}</b>
                </p>

                <p>
                    🔐 SHA-256 |
                    🔑 256-bit Dynamic Session |
                    ⛓ Dynamic Blockchain |
                    📋 Audit Logging
                </p>

            </div>
            """
        )
    )


    # =================================================================
    # SECURITY PANEL
    # =================================================================

    security_panel = HTML(

        f"""
        <div style="
            background:#eef6ff;
            border:2px solid #1f4e78;
            padding:15px;
            border-radius:10px;
            margin-top:15px;
            margin-bottom:15px;
        ">

            <h3>
                🔐 Active Security Session
            </h3>

            <b>User:</b>
            {CURRENT_USER}

            <br>

            <b>Authentication:</b>
            SHA-256

            <br>

            <b>Session Key:</b>
            256-bit dynamic

            <br>

            <b>Blockchain:</b>
            Active

            <br>

            <b>Audit Log:</b>
            Active

        </div>
        """
    )


    display(
        security_panel
    )


    # =================================================================
    # PATIENT INFORMATION
    # =================================================================

    display(
        HTML(
            "<h2>👤 Patient Information</h2>"
        )
    )


    patient_name = widgets.Text(

        description="Patient Name:",

        layout=widgets.Layout(
            width="550px"
        )
    )


    patient_age = widgets.Text(

        description="Age:",

        layout=widgets.Layout(
            width="400px"
        )
    )


    patient_contact = widgets.Text(

        description="Contact:",

        layout=widgets.Layout(
            width="550px"
        )
    )


    department = widgets.Text(

        description="Department:",

        layout=widgets.Layout(
            width="550px"
        )
    )


    gender = widgets.Dropdown(

        description="Gender:",

        options=[
            "",
            "Male",
            "Female",
            "Other"
        ],

        layout=widgets.Layout(
            width="400px"
        )
    )


    admission_date = widgets.Text(

        description="Admission Date:",

        placeholder="DD-MM-YYYY",

        layout=widgets.Layout(
            width="500px"
        )
    )


    doctor_name = widgets.Text(

        description="Doctor Name:",

        layout=widgets.Layout(
            width="550px"
        )
    )


    # =================================================================
    # MEDICINE INFORMATION
    # =================================================================

    display(
        HTML(
            "<h2>💊 Medicine Information</h2>"
        )
    )


    medicine_dropdown = widgets.Combobox(

        description="Medicine:",

        placeholder="Search medicine...",

        options=medicine_names,

        ensure_option=False,

        layout=widgets.Layout(
            width="650px"
        )
    )


    medicine_add = widgets.Button(

        description="ADD MEDICINE",

        button_style="success",

        icon="plus"
    )


    medicine_remove = widgets.Button(

        description="REMOVE",

        button_style="warning",

        icon="trash"
    )


    selected_medicines = widgets.SelectMultiple(

        description="Selected:",

        options=[],

        layout=widgets.Layout(
            width="650px",
            height="100px"
        )
    )


    medicine_dosage = widgets.Textarea(

        description="Dosage:",

        layout=widgets.Layout(
            width="700px",
            height="70px"
        )
    )


    medicine_side_effects = widgets.Textarea(

        description="Side-effects:",

        layout=widgets.Layout(
            width="700px",
            height="100px"
        )
    )


    medicine_used_for = widgets.Textarea(

        description="Used For:",

        layout=widgets.Layout(
            width="700px",
            height="100px"
        )
    )


    medicine_composition = widgets.Textarea(

        description="Composition:",

        layout=widgets.Layout(
            width="700px",
            height="80px"
        )
    )


    medicine_company = widgets.Text(

        description="Company:",

        layout=widgets.Layout(
            width="650px"
        )
    )


    medicine_brand = widgets.Text(

        description="Brand:",

        layout=widgets.Layout(
            width="650px"
        )
    )


    medicine_price = widgets.Text(

        description="Price:",

        layout=widgets.Layout(
            width="500px"
        )
    )


    sentiment_score = widgets.Text(

        description="Sentiment Score:",

        disabled=True,

        layout=widgets.Layout(
            width="500px"
        )
    )


    sentiment_label = widgets.Text(

        description="Sentiment Label:",

        disabled=True,

        layout=widgets.Layout(
            width="500px"
        )
    )


    matched_disease = widgets.Text(

        description="Matched Disease:",

        disabled=True,

        layout=widgets.Layout(
            width="600px"
        )
    )


    # =================================================================
    # MEDICINE AUTO-FILL
    # =================================================================

    def update_medicine_info(
            change
    ):

        if change["name"] != "value":

            return


        name = str(
            change["new"]
        ).strip()


        medicine_dosage.value = ""

        medicine_side_effects.value = ""

        medicine_used_for.value = ""

        medicine_composition.value = ""

        medicine_company.value = ""

        medicine_brand.value = ""

        medicine_price.value = ""

        sentiment_score.value = ""

        sentiment_label.value = ""

        matched_disease.value = ""


        if not name:

            return


        # -------------------------------------------------------------
        # JSON MEDICINE INFORMATION
        # -------------------------------------------------------------

        if name in medicine_dict:

            record = medicine_dict[
                name
            ]


            medicine_dosage.value = str(

                record.get(
                    "Dosage",
                    ""
                )
            )


            medicine_side_effects.value = str(

                record.get(
                    "Side-effects",

                    record.get(
                        "Side effects",

                        record.get(
                            "SideEffects",
                            ""
                        )
                    )
                )
            )


            medicine_used_for.value = str(

                record.get(
                    "Used For",

                    record.get(
                        "Used for",

                        record.get(
                            "UsedFor",
                            ""
                        )
                    )
                )
            )


            medicine_composition.value = str(

                record.get(
                    "Composition",
                    ""
                )
            )


            medicine_company.value = str(

                record.get(
                    "Company",
                    ""
                )
            )


            medicine_brand.value = str(

                record.get(
                    "Brand",

                    record.get(
                        "Brands",
                        ""
                    )
                )
            )


            medicine_price.value = str(

                record.get(
                    "Price",

                    record.get(
                        "Price (India)",
                        ""
                    )
                )
            )


        # -------------------------------------------------------------
        # SENTIMENT INFORMATION
        # -------------------------------------------------------------

        key = name.lower()


        if key in sentiment_dict:

            sentiment = sentiment_dict[
                key
            ]


            sentiment_score.value = str(

                sentiment.get(
                    "score",
                    ""
                )
            )


            sentiment_label.value = str(

                sentiment.get(
                    "label",
                    ""
                )
            )


            matched_disease.value = str(

                sentiment.get(
                    "matched_disease",
                    ""
                )
            )


    medicine_dropdown.observe(

        update_medicine_info,

        names="value"
    )


    # =================================================================
    # ADD MEDICINE
    # =================================================================

    medicine_selection = []


    def add_selected_medicine(
            button
    ):

        name = str(
            medicine_dropdown.value
        ).strip()


        if not name:

            return


        if name not in medicine_selection:

            medicine_selection.append(
                name
            )


        selected_medicines.options = (
            tuple(
                medicine_selection
            )
        )


    medicine_add.on_click(
        add_selected_medicine
    )


    # =================================================================
    # REMOVE MEDICINE
    # =================================================================

    def remove_selected_medicine(
            button
    ):

        selected = list(
            selected_medicines.value
        )


        for name in selected:

            if name in medicine_selection:

                medicine_selection.remove(
                    name
                )


        selected_medicines.options = (
            tuple(
                medicine_selection
            )
        )


    medicine_remove.on_click(
        remove_selected_medicine
    )


    # =================================================================
    # SYMPTOM / DISEASE SECTION
    # =================================================================

    display(
        HTML(
            "<h2>🩺 Symptoms and Clinical Information</h2>"
        )
    )


    disease_dropdown = widgets.Combobox(

        description="Disease:",

        placeholder="Search disease...",

        options=disease_names,

        ensure_option=False,

        layout=widgets.Layout(
            width="650px"
        )
    )


    symptom_text = widgets.Textarea(

        description="Symptoms:",

        layout=widgets.Layout(
            width="750px",
            height="120px"
        )
    )


    remedies_text = widgets.Textarea(

        description="Remedies:",

        layout=widgets.Layout(
            width="750px",
            height="120px"
        )
    )


    doctor_warning_text = widgets.Textarea(

        description="When to See Doctor:",

        layout=widgets.Layout(
            width="750px",
            height="120px"
        )
    )


    prevention_text = widgets.Textarea(

        description="Prevention Tips:",

        layout=widgets.Layout(
            width="750px",
            height="120px"
        )
    )


    # =================================================================
    # DISEASE AUTO-FILL
    # =================================================================

    def update_disease_info(
            change
    ):

        if change["name"] != "value":

            return


        disease = str(
            change["new"]
        ).strip()


        symptom_text.value = ""

        remedies_text.value = ""

        doctor_warning_text.value = ""

        prevention_text.value = ""


        if not disease:

            return


        if disease not in disease_dict:

            return


        record = disease_dict[
            disease
        ]


        symptom_text.value = str(

            record.get(
                "Symptoms",
                ""
            )
        )


        remedies_text.value = str(

            record.get(
                "Remedies",
                ""
            )
        )


        doctor_warning_text.value = str(

            record.get(

                "When to see doctors",

                record.get(

                    "When_to_see_doctors",

                    record.get(
                        "When to see doctors",
                        ""
                    )
                )
            )
        )


        prevention_text.value = str(

            record.get(

                "Prevention tips",

                record.get(
                    "Prevention_tips",
                    ""
                )
            )
        )


    disease_dropdown.observe(

        update_disease_info,

        names="value"
    )


    # =================================================================
    # PATIENT OUTPUT
    # =================================================================

    save_output = widgets.Output()


    # =================================================================
    # SAVE PATIENT BUTTON
    # =================================================================

    save_patient_button = widgets.Button(

        description="SAVE SECURE EHR",

        button_style="success",

        icon="save",

        layout=widgets.Layout(
            width="220px"
        )
    )


    # =================================================================
    # SAVE PATIENT FUNCTION
    # =================================================================

    def save_patient(
            button
    ):

        with save_output:

            clear_output()


            # ---------------------------------------------------------
            # BASIC VALIDATION
            # ---------------------------------------------------------

            if not patient_name.value.strip():

                print(
                    "❌ Patient Name is required."
                )

                return


            if not CURRENT_SESSION_KEY:

                print(
                    "❌ Security session is not active."
                )

                return


            # ---------------------------------------------------------
            # PATIENT RECORD
            # ---------------------------------------------------------

            patient_id = (

                str(
                    int(
                        time.time() * 1000
                    )
                )[-8:]
            )


            patient_record = {

                "patient_id":
                    patient_id,

                "patient_name":
                    patient_name.value.strip(),

                "age":
                    patient_age.value.strip(),

                "contact":
                    patient_contact.value.strip(),

                "department":
                    department.value.strip(),

                "gender":
                    gender.value,

                "admission_date":
                    admission_date.value.strip(),

                "doctor":
                    doctor_name.value.strip(),

                "disease":
                    disease_dropdown.value,

                "symptoms":
                    symptom_text.value,

                "medicines":
                    ", ".join(
                        medicine_selection
                    ),

                "dosage":
                    medicine_dosage.value,

                "used_for":
                    medicine_used_for.value,

                "composition":
                    medicine_composition.value,

                "company":
                    medicine_company.value,

                "brand":
                    medicine_brand.value,

                "side_effects":
                    medicine_side_effects.value,

                "price":
                    medicine_price.value,

                "sentiment_score":
                    sentiment_score.value,

                "sentiment_label":
                    sentiment_label.value,

                "matched_disease":
                    matched_disease.value,

                "remedies":
                    remedies_text.value,

                "when_to_see_doctor":
                    doctor_warning_text.value,

                "prevention_tips":
                    prevention_text.value,

                "created_by":
                    CURRENT_USER,

                "created_at":
                    now_string()
            }


            # ---------------------------------------------------------
            # EHR JSON
            # ---------------------------------------------------------

            ehr_json = json.dumps(

                patient_record,

                sort_keys=True,

                ensure_ascii=False
            )


            # ---------------------------------------------------------
            # SHA-256 EHR HASH
            # ---------------------------------------------------------

            ehr_hash = sha256_text(
                ehr_json
            )


            # ---------------------------------------------------------
            # DYNAMIC EHR ENCRYPTION
            # ---------------------------------------------------------

            encrypted_ehr = encrypt_ehr(

                ehr_json,

                CURRENT_SESSION_KEY
            )


            encrypted_hash = sha256_text(
                encrypted_ehr
            )


            # ---------------------------------------------------------
            # BLOCKCHAIN DATA
            # ---------------------------------------------------------

            blockchain_data = {

                "type":
                    "PATIENT_EHR",

                "patient_id":
                    patient_id,

                "patient_name":
                    patient_name.value.strip(),

                "disease":
                    disease_dropdown.value,

                "ehr_hash":
                    ehr_hash,

                "encrypted_ehr_hash":
                    encrypted_hash,

                "created_by":
                    CURRENT_USER,

                "timestamp":
                    now_string()
            }


            # ---------------------------------------------------------
            # CREATE BLOCK
            # ---------------------------------------------------------

            block = create_block(

                blockchain_data
            )


            # ---------------------------------------------------------
            # ADD BLOCKCHAIN INFORMATION
            # ---------------------------------------------------------

            patient_record[
                "ehr_hash"
            ] = ehr_hash


            patient_record[
                "encrypted_ehr_hash"
            ] = encrypted_hash


            patient_record[
                "block_index"
            ] = block[
                "index"
            ]


            patient_record[
                "previous_hash"
            ] = block[
                "previous_hash"
            ]


            patient_record[
                "blockchain_hash"
            ] = block[
                "hash"
            ]


            patient_records.append(
                patient_record
            )


            # ---------------------------------------------------------
            # AUDIT LOG
            # ---------------------------------------------------------

            add_audit_log(

                CURRENT_USER,

                "Added patient "
                +
                patient_id
            )


            # ---------------------------------------------------------
            # DISPLAY RESULT
            # ---------------------------------------------------------

            print(
                "=" * 90
            )

            print(
                "✅ SECURE EHR SAVED SUCCESSFULLY"
            )

            print(
                "=" * 90
            )

            print()

            print(
                "Patient ID:",
                patient_id
            )

            print()

            print(
                "Patient Name:",
                patient_name.value
            )

            print()

            print(
                "EHR SHA-256 Hash:"
            )

            print(
                ehr_hash
            )

            print()

            print(
                "Encrypted EHR Hash:"
            )

            print(
                encrypted_hash
            )

            print()

            print(
                "Blockchain Block Index:",
                block["index"]
            )

            print()

            print(
                "Previous Block Hash:"
            )

            print(
                block["previous_hash"]
            )

            print()

            print(
                "Current Blockchain Hash:"
            )

            print(
                block["hash"]
            )

            print()

            print(
                "Created By:",
                CURRENT_USER
            )

            print(
                "=" * 90
            )


    save_patient_button.on_click(
        save_patient
    )


    # =================================================================
    # CSV EXPORT
    # =================================================================

    csv_output = widgets.Output()


    export_button = widgets.Button(

        description="EXPORT PATIENT CSV",

        button_style="info",

        icon="download"
    )


    def export_csv(
            button
    ):

        with csv_output:

            clear_output()


            if not patient_records:

                print(
                    "❌ No newly created patient records."
                )

                return


            df = pd.DataFrame(
                patient_records
            )


            df.to_csv(

                PATIENT_FILE,

                index=False,

                encoding="utf-8"
            )


            print(
                "✅ Patient CSV generated:"
            )

            print(
                PATIENT_FILE
            )


            display(
                df
            )


            files.download(
                PATIENT_FILE
            )


    export_button.on_click(
        export_csv
    )


    # =================================================================
    # VIEW BLOCKCHAIN
    # =================================================================

    blockchain_output = widgets.Output()


    view_blockchain_button = widgets.Button(

        description="VIEW BLOCKCHAIN",

        button_style="warning",

        icon="link"
    )


    def view_blockchain(
            button
    ):

        with blockchain_output:

            clear_output()


            blockchain = load_blockchain()


            print(
                "=" * 100
            )

            print(
                "⛓ DYNAMIC BLOCKCHAIN"
            )

            print(
                "=" * 100
            )


            print()

            print(
                "Total Blocks:",
                len(blockchain)
            )


            for block in blockchain:

                print()

                print(
                    "-" * 100
                )

                print(
                    "Block Index:",
                    block.get(
                        "index"
                    )
                )

                print(
                    "Timestamp:",
                    block.get(
                        "timestamp"
                    )
                )

                print(
                    "Previous Hash:"
                )

                print(
                    block.get(
                        "previous_hash"
                    )
                )

                print()

                print(
                    "Current Hash:"
                )

                print(
                    block.get(
                        "hash"
                    )
                )

                print()

                print(
                    "Block Data:"
                )

                print(

                    json.dumps(

                        block.get(
                            "data",
                            {}
                        ),

                        indent=4,

                        ensure_ascii=False
                    )
                )

            print(
                "=" * 100
            )


    view_blockchain_button.on_click(
        view_blockchain
    )


    # =================================================================
    # VERIFY BLOCKCHAIN
    # =================================================================

    verify_output = widgets.Output()


    verify_button = widgets.Button(

        description="VERIFY BLOCKCHAIN",

        button_style="success",

        icon="check"
    )


    def verify_blockchain_button(
            button
    ):

        with verify_output:

            clear_output()


            valid, result_df = (
                verify_blockchain()
            )


            if valid:

                display(

                    HTML(

                        """
                        <div style="
                            background:#d9ead3;
                            padding:15px;
                            border-radius:8px;
                            border:2px solid #38761d;
                        ">

                        <h3>
                        ✅ BLOCKCHAIN VERIFIED
                        </h3>

                        All checked blocks have
                        valid hashes and valid
                        previous-hash links.

                        </div>
                        """
                    )
                )

            else:

                display(

                    HTML(

                        """
                        <div style="
                            background:#f4cccc;
                            padding:15px;
                            border-radius:8px;
                            border:2px solid #990000;
                        ">

                        <h3>
                        ⚠ BLOCKCHAIN INTEGRITY FAILURE
                        </h3>

                        One or more blocks failed
                        verification.

                        </div>
                        """
                    )
                )


            display(
                result_df
            )


    verify_button.on_click(
        verify_blockchain_button
    )


    # =================================================================
    # AUDIT LOG
    # =================================================================

    audit_output = widgets.Output()


    audit_button = widgets.Button(

        description="VIEW AUDIT LOG",

        button_style="warning",

        icon="history"
    )


    def view_audit_log(
            button
    ):

        with audit_output:

            clear_output()


            logs = load_audit_log()


            print(
                "=" * 90
            )

            print(
                "📋 SECURITY AUDIT LOG"
            )

            print(
                "=" * 90
            )


            if not logs:

                print(
                    "No audit records."
                )

                return


            audit_df = pd.DataFrame(
                logs
            )


            display(
                audit_df
            )


    audit_button.on_click(
        view_audit_log
    )


    # =================================================================
    # VIEW PATIENT RECORDS
    # =================================================================

    patient_output = widgets.Output()


    view_patient_button = widgets.Button(

        description="VIEW CURRENT PATIENTS",

        button_style="info",

        icon="users"
    )


    def view_patients(
            button
    ):

        with patient_output:

            clear_output()


            if not patient_records:

                print(
                    "No patient records created during this session."
                )

                return


            df = pd.DataFrame(
                patient_records
            )


            display(
                df
            )


    view_patient_button.on_click(
        view_patients
    )


    # =================================================================
    # DOWNLOAD BLOCKCHAIN
    # =================================================================

    download_blockchain_button = widgets.Button(

        description="DOWNLOAD BLOCKCHAIN",

        button_style="info"
    )


    download_blockchain_output = widgets.Output()


    def download_blockchain(
            button
    ):

        with download_blockchain_output:

            clear_output()


            print(
                "Preparing blockchain file..."
            )


            files.download(
                BLOCKCHAIN_FILE
            )


    download_blockchain_button.on_click(
        download_blockchain
    )


    # =================================================================
    # DOWNLOAD AUDIT LOG
    # =================================================================

    download_audit_button = widgets.Button(

        description="DOWNLOAD AUDIT LOG",

        button_style="info"
    )


    download_audit_output = widgets.Output()


    def download_audit(
            button
    ):

        with download_audit_output:

            clear_output()


            files.download(
                AUDIT_FILE
            )


    download_audit_button.on_click(
        download_audit
    )


    # =================================================================
    # LOGOUT
    # =================================================================

    logout_button = widgets.Button(

        description="LOGOUT",

        button_style="danger",

        icon="sign-out"
    )


    logout_output = widgets.Output()


    def logout(
            button
    ):

        global CURRENT_USER

        global CURRENT_SESSION_KEY

        global CURRENT_ENCRYPTION_KEY

        global CURRENT_TOKEN


        add_audit_log(

            CURRENT_USER,

            "logout"
        )


        CURRENT_USER = None

        CURRENT_SESSION_KEY = None

        CURRENT_ENCRYPTION_KEY = None

        CURRENT_TOKEN = None


        with logout_output:

            clear_output()


            print(
                "✅ Logout successful."
            )

            print()

            print(
                "Run the program cell again to open the login window."
            )


    logout_button.on_click(
        logout
    )


    # =================================================================
    # DISPLAY PATIENT FORM
    # =================================================================

    display(

        widgets.VBox([

            patient_name,

            patient_age,

            patient_contact,

            department,

            gender,

            admission_date,

            doctor_name

        ])
    )


    # =================================================================
    # DISPLAY MEDICINE FORM
    # =================================================================

    display(

        widgets.VBox([

            medicine_dropdown,

            widgets.HBox([

                medicine_add,

                medicine_remove

            ]),

            selected_medicines,

            medicine_composition,

            medicine_company,

            medicine_brand,

            medicine_used_for,

            medicine_dosage,

            medicine_side_effects,

            medicine_price,

            sentiment_score,

            sentiment_label,

            matched_disease

        ])
    )


    # =================================================================
    # DISPLAY CLINICAL INFORMATION
    # =================================================================

    display(

        widgets.VBox([

            disease_dropdown,

            symptom_text,

            remedies_text,

            doctor_warning_text,

            prevention_text

        ])
    )


    # =================================================================
    # SAVE / EXPORT
    # =================================================================

    display(

        widgets.HBox([

            save_patient_button,

            export_button

        ])
    )


    display(
        save_output
    )


    display(
        csv_output
    )


    # =================================================================
    # SECURITY OPERATIONS
    # =================================================================

    display(

        HTML(
            "<h2>⛓ Blockchain and Security</h2>"
        )
    )


    display(

        widgets.HBox([

            view_blockchain_button,

            verify_button,

            audit_button

        ])
    )


    display(
        blockchain_output
    )

    display(
        verify_output
    )

    display(
        audit_output
    )


    # =================================================================
    # PATIENT RECORD OPERATIONS
    # =================================================================

    display(

        HTML(
            "<h2>📊 Patient and File Operations</h2>"
        )
    )


    display(

        widgets.HBox([

            view_patient_button,

            download_blockchain_button,

            download_audit_button,

            logout_button

        ])
    )


    display(
        patient_output
    )

    display(
        download_blockchain_output
    )

    display(
        download_audit_output
    )

    display(
        logout_output
    )


# =====================================================================
# 30. START SYSTEM
# =====================================================================

initialize_blockchain()


print()
print("=" * 80)
print("SYSTEM INITIALIZED")
print("=" * 80)

print()
print(
    "Medicine records loaded:",
    len(medicines)
)

print(
    "Disease/Symptom records loaded:",
    len(symptoms)
)

print(
    "Medicine sentiment records loaded:",
    len(sentiment_df)
)

print(
    "Existing blockchain blocks:",
    len(load_blockchain())
)

print(
    "Existing audit records:",
    len(load_audit_log())
)

print()
print("LOGIN CREDENTIALS")
print("-" * 80)

print(
    "Doctor:"
)

print(
    "Username: doctor1"
)

print(
    "Password: Doctor@123"
)

print()

print(
    "Administrator:"
)

print(
    "Username: admin"
)

print(
    "Password: Admin@123"
)

print()
print("=" * 80)
print("Enter the credentials in the Login window above.")
print("=" * 80)

DYNAMIC SECURE HOSPITAL EHR

Upload your existing hospital files.

Required:
1. blockchain_db JSON
2. audit_log JSON
3. Medicine JSON
4. Symptoms JSON
5. Medicine sentiment CSV

